# BharatStock AI — AIML Model Research (V92)

This notebook compares the original 8 features with the expanded 16 features, evaluates classical ML models on chronological walk-forward folds, and optionally trains LSTM and GRU sequence models. It does **not** change or automatically deploy the live API model.

**Target:** next-session direction, `Close[t+1] > Close[t]`. This is an academic research experiment, not a trading recommendation. Historical accuracy is not a promise of future performance.


In [ ]:
!pip -q install yfinance scikit-learn pandas numpy matplotlib torch


In [ ]:
import sys, os, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## 1. Load historical OHLCV data

Choose a Yahoo Finance ticker such as `RELIANCE.NS`, `TCS.NS`, `INFY.NS`, or `^NSEI`. You can also upload a CSV with Date, Open, High, Low, Close, Volume columns.


In [ ]:
import yfinance as yf
SYMBOL='RELIANCE.NS'
raw=yf.Ticker(SYMBOL).history(period='5y', auto_adjust=True)
raw=raw.rename(columns={c:str(c).title() for c in raw.columns})
raw=raw[['Open','High','Low','Close','Volume']].dropna()
print(raw.shape, raw.tail(3))


## 2. Causal feature engineering

All predictors use current or past OHLCV rows. The final row has no known next-session label and is excluded.


In [ ]:
BASE=['RSI','MACD_pct','MACD_Signal_pct','SMA20_gap','SMA50_gap','EMA20_gap','ATR_pct','Return_5']
EXTRA=['Return_1','Return_10','Return_20','Volatility_5','Range_pct','Close_Location','Gap_pct','Volume_Ratio_20']

def make_features(raw):
    d=raw.copy().sort_index(); c,h,l,o,v=(pd.to_numeric(d[k],errors='coerce') for k in ['Close','High','Low','Open','Volume'])
    d['SMA20']=c.rolling(20).mean(); d['SMA50']=c.rolling(50).mean(); d['EMA20']=c.ewm(span=20,adjust=False).mean()
    delta=c.diff(); gain=delta.clip(lower=0).rolling(14).mean(); loss=-delta.clip(upper=0).rolling(14).mean()
    d['RSI']=(100-100/(1+gain/loss.replace(0,np.nan))).fillna(50).clip(0,100)
    macd=c.ewm(span=12,adjust=False).mean()-c.ewm(span=26,adjust=False).mean()
    d['MACD_pct']=macd/c.replace(0,np.nan); d['MACD_Signal_pct']=macd.ewm(span=9,adjust=False).mean()/c.replace(0,np.nan)
    d['SMA20_gap']=c/d['SMA20']-1; d['SMA50_gap']=c/d['SMA50']-1; d['EMA20_gap']=c/d['EMA20']-1
    prev=c.shift(1); tr=pd.concat([h-l,(h-prev).abs(),(l-prev).abs()],axis=1).max(axis=1)
    d['ATR_pct']=tr.rolling(14).mean()/c.replace(0,np.nan); d['Return_5']=c.pct_change(5)
    d['Return_1']=c.pct_change(); d['Return_10']=c.pct_change(10); d['Return_20']=c.pct_change(20)
    d['Volatility_5']=c.pct_change().rolling(5).std(); d['Range_pct']=(h-l)/c.replace(0,np.nan)
    d['Close_Location']=((c-l)/(h-l).replace(0,np.nan)).clip(0,1); d['Gap_pct']=o/prev.replace(0,np.nan)-1
    d['Volume_Ratio_20']=v/v.rolling(20).median().replace(0,np.nan)
    d['Target']=(c.shift(-1)>c).astype(float); d.loc[d.index[-1],'Target']=np.nan
    return d.replace([np.inf,-np.inf],np.nan)

data=make_features(raw)
print('Feature count:',len(BASE+EXTRA),'Rows with complete features:',data[BASE+EXTRA+['Target']].dropna().shape[0])


## 3. Classical ML feature ablation + model comparison

Each feature set uses identical expanding-window test periods. The final training row before each test fold is purged because its label depends on the first test row. Model configurations are fixed in advance; this is a conservative comparison, not exhaustive hyperparameter search.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss, log_loss

FEATURE_SETS={'Original 8':BASE,'Expanded 16':BASE+EXTRA,'Momentum & volatility':['RSI','Return_1','Return_5','Return_10','Return_20','Volatility_5','ATR_pct'],'Trend only':['MACD_pct','MACD_Signal_pct','SMA20_gap','SMA50_gap','EMA20_gap']}
FACTORIES={
 'LogisticRegression':lambda:make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,class_weight='balanced',C=.1,random_state=42)),
 'RandomForest':lambda:RandomForestClassifier(n_estimators=180,max_depth=6,min_samples_leaf=8,max_features='sqrt',class_weight='balanced_subsample',random_state=42,n_jobs=-1),
 'ExtraTrees':lambda:ExtraTreesClassifier(n_estimators=180,max_depth=6,min_samples_leaf=8,max_features='sqrt',class_weight='balanced',random_state=43,n_jobs=-1),
 'HistGradientBoosting':lambda:HistGradientBoostingClassifier(max_iter=120,learning_rate=.04,max_leaf_nodes=7,l2_regularization=2,random_state=44)}

def walk_forward(df, features, n_folds=5):
    z=df[features+['Target']].dropna(); n=len(z); initial=max(100,int(n*.5)); remain=n-initial
    if remain<n_folds*12: return []
    sizes=np.full(n_folds,remain//n_folds,dtype=int); sizes[:remain%n_folds]+=1; cursor=initial; rows=[]
    for fold,size in enumerate(sizes,1):
        tr=z.iloc[:cursor-1]; te=z.iloc[cursor:cursor+int(size)]; cursor+=int(size)
        if len(tr)<80 or te.empty: continue
        ytr=tr.Target.astype(int); yte=te.Target.astype(int); base=int(ytr.value_counts().idxmax())
        row={'fold':fold,'baseline_accuracy':accuracy_score(yte,np.full(len(te),base))}
        for name,make in FACTORIES.items():
            m=make(); m.fit(tr[features],ytr); pred=m.predict(te[features])
            row[name]={'accuracy':accuracy_score(yte,pred),'balanced_accuracy':balanced_accuracy_score(yte,pred),'f1':f1_score(yte,pred,zero_division=0)}
        rows.append(row)
    return rows

results={}
for fs,features in FEATURE_SETS.items():
    folds=walk_forward(data,features)
    results[fs]=folds
    print('\n',fs,'folds=',len(folds))
    for model in FACTORIES:
        vals=[r[model]['accuracy'] for r in folds]
        print(model, 'mean accuracy=',round(float(np.mean(vals)),4) if vals else None)
    b=[r['baseline_accuracy'] for r in folds]
    print('Majority baseline=',round(float(np.mean(b)),4) if b else None)


In [ ]:
# Plot mean walk-forward accuracy by feature set and model
summary=[]
for fs,folds in results.items():
    for model in list(FACTORIES)+['Majority baseline']:
        vals=[r[model]['accuracy'] if model!='Majority baseline' else r['baseline_accuracy'] for r in folds]
        summary.append({'Feature set':fs,'Model':model,'Mean accuracy':np.mean(vals) if vals else np.nan})
plot_df=pd.DataFrame(summary)
if not plot_df.empty:
    pivot=plot_df.pivot(index='Feature set',columns='Model',values='Mean accuracy')
    pivot.plot(kind='bar',figsize=(12,5),ylim=(0,1),title='Chronological walk-forward accuracy')
    plt.ylabel('Accuracy'); plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()


## 4. LSTM and GRU sequence models (optional)

The sequence models are trained separately inside each chronological fold. The scaler is fitted only on the training segment. Early stopping is not used on the outer test fold. The last part of the training window is used as an internal chronological validation segment for epoch selection.


In [ ]:
class SeqNet(nn.Module):
    def __init__(self, kind, n_features, hidden=32):
        super().__init__(); self.kind=kind
        self.rnn=(nn.LSTM(n_features,hidden,batch_first=True) if kind=='LSTM' else nn.GRU(n_features,hidden,batch_first=True))
        self.head=nn.Sequential(nn.Dropout(.2),nn.Linear(hidden,1))
    def forward(self,x):
        out,_=self.rnn(x); return self.head(out[:,-1,:]).squeeze(-1)

def seq_arrays(x,y,lookback):
    xs=[]; ys=[]
    for end in range(lookback-1,len(x)):
        xs.append(x[end-lookback+1:end+1]); ys.append(y[end])
    return np.asarray(xs,dtype=np.float32),np.asarray(ys,dtype=np.float32)

def fit_sequence_fold(train_df,test_df,features,kind,lookback=20,epochs=25):
    scaler=StandardScaler().fit(train_df[features]) # training rows only
    xtr=scaler.transform(train_df[features]); ytr=train_df.Target.to_numpy(dtype=np.float32)
    xte=scaler.transform(test_df[features]); yte=test_df.Target.to_numpy(dtype=np.int64)
    X,Y=seq_arrays(xtr,ytr,lookback); Xtest,Ytest=seq_arrays(np.vstack([xtr[-(lookback-1):],xte]),np.r_[ytr[-(lookback-1):],yte],lookback)
    if len(X)<80 or len(Xtest)==0: return None
    split=max(1,int(len(X)*.85)); Xfit,Yfit=X[:split],Y[:split]; Xval,Yval=X[split:],Y[split:]
    if len(Xval)==0: Xval,Yval=Xfit[-1:],Yfit[-1:]
    model=SeqNet(kind,X.shape[-1]).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=.001,weight_decay=1e-4); lossfn=nn.BCEWithLogitsLoss()
    loader=DataLoader(TensorDataset(torch.tensor(Xfit),torch.tensor(Yfit)),batch_size=32,shuffle=False)
    best=float('inf'); best_state=None; patience=5; stale=0
    for epoch in range(epochs):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=lossfn(model(xb),yb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad(): val=float(lossfn(model(torch.tensor(Xval).to(DEVICE)),torch.tensor(Yval).to(DEVICE)).item())
        if val<best: best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): probs=torch.sigmoid(model(torch.tensor(Xtest).to(DEVICE))).cpu().numpy()
    pred=(probs>=.5).astype(int)
    return {'accuracy':accuracy_score(Ytest,pred),'balanced_accuracy':balanced_accuracy_score(Ytest,pred),'f1':f1_score(Ytest,pred,zero_division=0),'test_rows':len(Ytest),'epochs_run':epoch+1}

# Use the expanded feature set; each test fold is still strictly chronological.
seq_results={'LSTM':[],'GRU':[]}
features=BASE+EXTRA; z=data[features+['Target']].dropna(); n=len(z); initial=max(100,int(n*.5)); remaining=n-initial; nf=5
if remaining>=nf*12:
    sizes=np.full(nf,remaining//nf,dtype=int); sizes[:remaining%nf]+=1; cursor=initial
    for fold,size in enumerate(sizes,1):
        tr=z.iloc[:cursor-1]; te=z.iloc[cursor:cursor+int(size)]; cursor+=int(size)
        if len(tr)<120 or te.empty: continue
        for kind in ['LSTM','GRU']:
            try:
                result=fit_sequence_fold(tr,te,features,kind)
                if result: result['fold']=fold; seq_results[kind].append(result)
            except Exception as e: print(kind,'fold',fold,'skipped:',repr(e))
for kind,folds in seq_results.items():
    print(kind,'folds',len(folds),'mean accuracy',np.mean([x['accuracy'] for x in folds]) if folds else 'not enough data')


## 5. Interpretation and export

Do not claim that one model is superior from one ticker or a small number of folds. Compare against the majority baseline, balanced accuracy, and variability across folds. LSTM/GRU may underperform classical models on small daily datasets. A model should not be selected based on the final holdout repeatedly. A future-period test is required after selection.


In [ ]:
final_report={'symbol':SYMBOL,'target':'next-session close direction','device':DEVICE,'classical_feature_ablation':results,'sequence_models':seq_results,'caveats':['Historical results are not future guarantees.','No transaction costs, slippage, taxes or execution simulation in this classification benchmark.','Sequence models are research candidates only.','Do not interpret sigmoid/model scores as calibrated trade-success probabilities.'] }
with open('bharatstock_ai_research_results.json','w') as f: json.dump(final_report,f,indent=2,default=lambda x: float(x) if isinstance(x,(np.floating,)) else int(x) if isinstance(x,(np.integer,)) else x)
print('Saved bharatstock_ai_research_results.json')
